# Tutorial for training a nest detection model

### Install Comet ML

In [1]:
!pip install comet_ml

In [2]:
%load_ext autoreload
%autoreload 2

### Install DeepForest library

In [3]:
# !git clone https://github.com/weecology/DeepForest.git

In [4]:
#%cd DeepForest
!pip install -e .
#%cd ..

Obtaining file:///home/cwinkelmann/work/deepforest
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for deepforest (pyproject.toml) ... done
  Created wheel for deepforest: filename=deepforest-1.5.3.dev0-0.editable-py3-none-any.whl size=6956 sha256=00f4fa38fbade7bb74a556b0995d0d9594d9d6b879f9aa466bc192039848a579
  Stored in directory: /tmp/pip-ephem-wheel-cache-7k9dyjs8/wheels/0e/87/49/bdac46f41eeebe85c9808e3df878ff826df7893c8aac0fd74c
Successfully built deepforest
  Attempting uninstall: deepforest
    Found existing installation: deepforest 1.3.3
    Can't uninstall 'deepforest'. No files were found to uninstall.


In [5]:
!pwd

/home/cwinkelmann/work/deepforest


In [6]:
import os
import sys

# deepforest_path = os.path.abspath("DeepForest")
# deepforest_path

In [7]:
# if deepforest_path not in sys.path:
#     sys.path.insert(0, deepforest_path)

In [3]:
# load the modules
import comet_ml
import os
import time
import numpy as np
import pandas as pd
import torch
from deepforest import main
from deepforest import get_data
from deepforest import utilities
from deepforest import preprocess
from tqdm import tqdm
from pytorch_lightning.loggers import CometLogger
import zipfile
import matplotlib.pyplot as plt
import subprocess

/home/cwinkelmann/miniconda3/envs/DeepForest_CWFork/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Set up Environment Variables

#### In Google Colab
Use Colab's secret storage to securely store your API key.

1. Locate the `Secrets` tab on the left-hand side panel in your Colab notebook.
2. Add a new secret with the key name as `COMET_API_KEY` and paste your Comet ML API key as the value.

#### Locally
Set an environment variable `COMET_API_KEY` in your operating system.

##### Windows
1. Open Command Prompt and set the environment variable:

    ```bash
    setx COMET_API_KEY "your_comet_api_key"
    ```

2. Restart your terminal or IDE.

##### macOS/Linux
1. Open your terminal and add the following line to your `.bashrc`, `.zshrc`, or `.profile` file:

    ```bash
    export COMET_API_KEY="your_comet_api_key"
    ```

2. Save the file and reload the shell configuration:

    ```bash
    source ~/.bashrc  # or ~/.zshrc, ~/.profile, etc.
    ```

In [9]:
PLATFORM = "local"  # Platform can be colab or local
environment = {}
if PLATFORM == "colab":
    from google.colab import userdata

    environment["api_key"] = userdata.get("COMET_API_KEY")
else:
    environment["api_key"] = os.getenv("COMET_API_KEY")

In [4]:
api_key = "bVOa3vnaXoP7OIstSDdblokzb"

In [5]:
# change the project_name
comet_logger = CometLogger(project="temporary2", api_key=api_key)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/karisu/temporary2/9b2e7357598e4513827182cb5663a647

COMET INFO: Couldn't find a Git repository in '/home/cwinkelmann/work/deepforest' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


### Download the Bird nest dataset

In [6]:
from pathlib import Path
extract_folder = Path("/home/cwinkelmann/work/deepforest/data/nest")

In [7]:
# Check if the annotations file has been extracted from the zip file
annotations = pd.read_csv(os.path.join(extract_folder, "nest_data.csv"))
annotations.head()

,image_path,xmin,ymin,xmax,ymax,label,annotator
0,JetPortNew_03_029_2022_DJI_0332.JPG,5414.798306,943.386925,5506.271111,1032.841833,Nest,NaN
1,JetPortNew_03_029_2022_DJI_0332.JPG,5713.079191,887.063465,5819.797463,977.181001,Nest,NaN
2,JetPortNew_03_029_2022_DJI_0332.JPG,5651.434474,758.513449,5790.632221,865.859338,Nest,NaN
3,JetPortNew_03_029_2022_DJI_0332.JPG,5405.787642,348.183777,5508.558036,444.584273,Nest,NaN
4,JetPortNew_03_029_2022_DJI_0332.JPG,5290.340722,235.490240,5386.772855,338.679503,Nest,NaN


In [8]:
# Gather all the images ending with .JPG
image_names = [file for file in os.listdir(extract_folder) if file.endswith(".JPG")]
image_names

['Jerrod_03_21_2022DJI_0140.JPG',
 'Jerrod_03_21_2022DJI_0035.JPG',
 'Horus_04_27_2022_DJI_0286.JPG',
 'Jerrod_03_21_2022DJI_0060.JPG',
 'Jerrod_03_21_2022DJI_0230.JPG',
 'Horus_04_27_2022_DJI_0290.JPG',
 'Jerrod_03_21_2022DJI_0184.JPG',
 'JetPortNew_03_029_2022_DJI_0424.JPG',
 'Jerrod_03_21_2022DJI_0198.JPG',
 'JetPortNew_03_029_2022_DJI_0320.JPG',
 'JetPortNew_03_029_2022_DJI_0434.JPG',
 'Jerrod_03_21_2022DJI_0182.JPG',
 'Jerrod_03_21_2022DJI_0059.JPG',
 'Horus_04_27_2022_DJI_0323.JPG',
 'Horus_04_27_2022_DJI_0208.JPG',
 'Horus_04_27_2022_DJI_0340.JPG',
 'Horus_04_27_2022_DJI_0249.JPG',
 'JetPortNew_03_029_2022_DJI_0277.JPG',
 'JetPortNew_03_029_2022_DJI_0092.JPG',
 'JetPortNew_03_029_2022_DJI_0179.JPG',
 'Horus_04_27_2022_DJI_0276.JPG',
 'Horus_04_27_2022_DJI_0324.JPG',
 'JetPortNew_03_029_2022_DJI_0023.JPG',
 'Jerrod_03_21_2022DJI_0232.JPG',
 'Horus_04_27_2022_DJI_0279.JPG',
 'JetPortNew_03_029_2022_DJI_0170.JPG',
 'JetPortNew_03_029_2022_DJI_0089.JPG',
 'JetPortNew_03_029_2022_DJI

In [9]:
# Generate crops of the image which has Region of Interest (ROI)
crop_dir = os.path.join(os.getcwd(), "train_data_folder", "images")
Path(crop_dir).mkdir(parents=True, exist_ok=True)

In [37]:
%%capture


annotation_path = os.path.join(extract_folder, "nest_data.csv")
all_annotations = []
for image in image_names:
    image_path = os.path.join(extract_folder, image)
    print(f"Processing image: {image_path}")
    annotations = preprocess.split_raster(
        path_to_raster=image_path,
        annotations_file=annotation_path,
        patch_size=640,
        patch_overlap=0,
        save_dir=crop_dir,
    )
    print(f"cRopping to: {crop_dir}")
    all_annotations.append(annotations)


train_annotations = pd.concat(all_annotations, ignore_index=True)

2025-07-16 21:40:16.655 | INFO     | deepforest.preprocess:split_raster:52 - writeing 1 annotations for Jerrod_03_21_2022DJI_0140.JPG to /home/cwinkelmann/work/deepforest/train_data_folder/images
2025-07-16 21:40:17.421 | INFO     | deepforest.preprocess:split_raster:52 - writeing 2 annotations for Jerrod_03_21_2022DJI_0035.JPG to /home/cwinkelmann/work/deepforest/train_data_folder/images
2025-07-16 21:40:17.946 | INFO     | deepforest.preprocess:split_raster:52 - writeing 6 annotations for Horus_04_27_2022_DJI_0286.JPG to /home/cwinkelmann/work/deepforest/train_data_folder/images
2025-07-16 21:40:18.760 | INFO     | deepforest.preprocess:split_raster:52 - writeing 9 annotations for Jerrod_03_21_2022DJI_0060.JPG to /home/cwinkelmann/work/deepforest/train_data_folder/images
2025-07-16 21:40:19.965 | INFO     | deepforest.preprocess:split_raster:52 - writeing 3 annotations for Jerrod_03_21_2022DJI_0230.JPG to /home/cwinkelmann/work/deepforest/train_data_folder/images
2025-07-16 21:40:20.

In [10]:
image_paths = train_annotations.image_path.unique()

# split into 70% train, 20% validation and 10% test annotations
temp_paths = np.random.choice(image_paths, int(len(image_paths) * 0.30))
valid_paths = np.random.choice(temp_paths, int(len(image_paths) * 0.20))
test_paths = [path for path in temp_paths if path not in valid_paths]

valid_annotations = train_annotations.loc[train_annotations.image_path.isin(valid_paths)]
test_annotations = train_annotations.loc[train_annotations.image_path.isin(test_paths)]
train_annotations = train_annotations.loc[~train_annotations.image_path.isin(temp_paths)]

NameError: name 'train_annotations' is not defined

In [11]:
# View output
# print(train_annotations.head())
#print("There are {} training crown annotations".format(train_annotations.shape[0]))
#print("There are {} test crown annotations".format(valid_annotations.shape[0]))

# save to file and create the file dir
annotations_file = os.path.join(crop_dir, "train.csv")
validation_file = os.path.join(crop_dir, "valid.csv")
test_file = os.path.join(crop_dir, "test.csv")

# Write window annotations file without a header row, same location as the "base_dir" above.
train_annotations.to_csv(annotations_file, index=False)
valid_annotations.to_csv(validation_file, index=False)
test_annotations.to_csv(test_file, index=False)

NameError: name 'train_annotations' is not defined

In [33]:

# Michigan birds

# initialize the model and change the corresponding config file
m = main.deepforest(label_dict={"Bird": 0})

# move to GPU and use all the GPU resources
m.config["gpus"] = "-1"
m.config["batch_size"] = 64
m.config["train"]["csv_file"] = "/home/cwinkelmann/work/deepforest/data/michigan/michigan_train.csv"
m.config["train"]["root_dir"] = "/home/cwinkelmann/work/deepforest/data/michigan/"

# Define the learning scheduler type
m.config["train"]["scheduler"]["type"] = "cosine"
m.config["score_thresh"] = 0.4
m.config["train"]["epochs"] = 20
m.config["validation"]["csv_file"] = "/home/cwinkelmann/work/deepforest/data/michigan/michigan_test.csv"
m.config["validation"]["root_dir"] = "/home/cwinkelmann/work/deepforest/data/michigan/"
m.config["validation"]["val_accuracy_interval"] = 2


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [34]:
# ## Tutorial Case
# # initialize the model and change the corresponding config file
# m = main.deepforest(label_dict={"Nest": 0})
#
# # move to GPU and use all the GPU resources
# m.config["gpus"] = "-1"
# m.config["batch_size"] = 64
# m.config["train"]["csv_file"] = annotations_file
# m.config["train"]["root_dir"] = os.path.dirname(annotations_file)
#
# # Define the learning scheduler type
# m.config["train"]["scheduler"]["type"] = "cosine"
# m.config["score_thresh"] = 0.4
# m.config["train"]["epochs"] = 100
# m.config["validation"]["csv_file"] = validation_file
# m.config["validation"]["root_dir"] = os.path.dirname(validation_file)
# m.config["validation"]["val_accuracy_interval"] = 2

In [35]:
m.config["train"]["scheduler"]["type"]

'cosine'

In [36]:
# create a pytorch lighting trainer used to training
# Disable the sanity check for validation data
comet_logger = CometLogger(project="temporary2", api_key=api_key, name="Michigan_birds")
m.create_trainer(logger=comet_logger, num_sanity_val_steps=0)
# load the lastest release model (RetinaNet)
m.load_model(model_name='weecology/deepforest-bird', label_dict={"Bird": 0})



COMET INFO: An experiment with the same configuration options is already running and will be reused.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experiment

In [37]:
# Start the training
start_time = time.time()
m.trainer.fit(m)
print(f"--- Training on GPU: {(time.time() - start_time):.2f} seconds ---")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                 | Type                  | Params | Mode 
-----------------------------------------------------------------------
0 | model                | RetinaNet             | 32.1 M | train
1 | iou_metric           | IntersectionOverUnion | 0      | train
2 | mAP_metric           | MeanAveragePrecision  | 0      | train
3 | empty_frame_accuracy | BinaryAccuracy        | 0      | train
-----------------------------------------------------------------------
31.9 M    Trainable params
222 K     Non-trainable params
32.1 M    Total params
128.592   Total estimated model params size (MB)
205       Modules in train mode
0         Modules in eval mode
/home/cwinkelmann/miniconda3/envs/DeepForest_CWFork/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Epoch 0: 100%|██████████| 76/76 [04:10<00:00,  0.30it/s, v_num=daa6]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 11/11 [00:40<00:00,  0.27it/s]

/home/cwinkelmann/work/deepforest/src/deepforest/evaluate.py:103: UserWarning: Converting predictions to GeoDataFrame using geometry column
  warnings.warn("Converting predictions to GeoDataFrame using geometry column")
/home/cwinkelmann/work/deepforest/src/deepforest/IoU.py:114: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  iou_df = pd.concat(iou_df)
/home/cwinkelmann/work/deepforest/src/deepforest/IoU.py:114: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  iou_df = pd.concat(iou_df)
/home/cwinke


Epoch 1: 100%|██████████| 76/76 [04:11<00:00,  0.30it/s, v_num=daa6]    
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:   9%|▉         | 1/11 [00:02<00:20,  0.49it/s]

/home/cwinkelmann/miniconda3/envs/DeepForest_CWFork/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


...


Validation DataLoader 0:  73%|███████▎  | 8/11 [00:30<00:11,  0.26it/s]

/home/cwinkelmann/miniconda3/envs/DeepForest_CWFork/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)



Epoch 19: 100%|██████████| 76/76 [04:50<00:00,  0.26it/s, v_num=daa6]   

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 76/76 [04:50<00:00,  0.26it/s, v_num=daa6]
--- Training on GPU: 6135.37 seconds ---


In [38]:
comet_logger.experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : Michigan_birds
COMET INFO:     url                   : https://www.comet.com/karisu/temporary2/ab9ac54001b74ef7a36181d79d07daa6
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     Bird_Precision                   : 1.0
COMET INFO:     Bird_Recall                      : 1.0
COMET INFO:     box_precision [10]               : (0.4243277609348297, 0.5334511399269104)
COMET INFO:     box_recall [10]                  : (0.7812743186950684, 0.8226668834686279)
COMET INFO:     iou [10]                         : (0.6319712996482849, 0.6624354720115662)
COMET INFO:     iou/cl_0 [10]                    : (0.6319713592529297, 0.662436306476593)
COMET INFO:

In [17]:
# save the prediction result to a prediction folder
save_dir = os.path.join(os.getcwd(), "pred_result_test")
results = m.evaluate(test_file,
                     root_dir = os.path.dirname(test_file),
                     iou_threshold=0.4,
                     savedir=save_dir)

TypeError: deepforest.evaluate() got an unexpected keyword argument 'savedir'

In [ ]:
results["box_precision"]

In [ ]:
results["box_recall"]

In [30]:
# save the results to a csv file
results["results"].to_csv("results_test_lr_cosine.csv", index=False)

In [ ]:
# Save the model checkpoint
m.trainer.save_checkpoint(
    os.path.join(root_folder, "checkpoint_epochs_10_cosine_lr_retinanet.pl"))

In [ ]:
torch.save(m.model.state_dict(), os.path.join(root_folder, "weights_cosine_lr"))

In [ ]:
# Load from the saved checkpoint
model = main.deepforest.load_from_checkpoint(
    os.path.join(root_folder, "checkpoint_epochs_10_cosine_lr_retinanet.pl"))

In [ ]:
# Add a path to an image to test the model on
path = ""
predicted_image = model.predict_tile(path=path,
                                      return_plot=True,
                                      patch_size=300,
                                      patch_overlap=0.25)
plt.imshow(predicted_image)
plt.show()